# Week 5, Embeddings: A Map of Meaning  (completion problem, **skeleton** version)

**The heart of the course.** You'll turn each item, a passage or a picture, into a *vector*, a
few hundred numbers whose position is learned from the company it keeps.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q sentence-transformers scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Load a sentence-embedding model

`sentence-transformers` downloads a small model (`all-MiniLM-L6-v2`) the first time.

In [ ]:
def get_text_embedder():
    """Return a function texts->vectors. Real model when possible; stand-in offline."""
    if not SMOKE:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("all-MiniLM-L6-v2")
            print("using real model: all-MiniLM-L6-v2")
            return lambda texts: np.asarray(model.encode(list(texts), normalize_embeddings=True))
        except Exception as e:
            print("could not load real model, using offline stand-in:", e)
    # Offline stand-in: hashed bag-of-words -> deterministic vector. NOT a real embedding;
    # it only lets the plotting/clustering code run in the compatibility harness.
    print("using OFFLINE STAND-IN embedder (harness only, not real embeddings)")
    rng_dim = 64
    def embed(texts):
        out = np.zeros((len(texts), rng_dim))
        for i, t in enumerate(texts):
            for tok in t.lower().split():
                out[i, hash(tok) % rng_dim] += 1.0
        norms = np.linalg.norm(out, axis=1, keepdims=True)
        return out / np.where(norms == 0, 1, norms)
    return embed

embed_text = get_text_embedder()

## Embed your own corpus

In class this is *your* data, a beloved artist's songs, fan posts, museum blurbs.

In [ ]:
corpus = [
    # "space" cluster
    "the rocket lifted off toward the distant red planet",
    "astronauts floated in orbit above the blue earth",
    "a telescope spotted a new moon around saturn",
    # "cooking" cluster
    "fold the butter into the flour until crumbly",
    "simmer the tomatoes with garlic and fresh basil",
    "whisk the eggs then pour the batter into the pan",
    # "heartbreak" cluster
    "you left and the silence in the house got loud",
    "i kept your number but i never call it now",
    "we were a song that the radio stopped playing",
]
vecs = embed_text(corpus)
print("embedded", len(corpus), "items into", vecs.shape[1], "dimensions")

### Or better: your own corpus

The demo corpus above is training wheels. If you collected your corpus in Week 4 (it saved as
`data.csv` in your Drive project folder), the cell below swaps it in, and everything downstream,
the maps, the clusters, the search, runs on *your* data. If it isn't there yet, the demo corpus
is fine for today.

In [ ]:
# Week 4 saved your corpus to the Drive project folder; embed its text column.
_corpus_file = os.path.join(PROJECT_DIR, "data.csv")
if os.path.exists(_corpus_file):
    import pandas as pd
    _df = pd.read_csv(_corpus_file)
    _col = next((c for c in _df.columns if _df[c].dtype == object), None)  # or set _col = "your_column"
    corpus = _df[_col].dropna().astype(str).head(200).tolist()
    vecs = embed_text(corpus)
    print(f"embedded YOUR corpus: {len(corpus)} items (text column {_col!r})")
else:
    print("no data.csv in your project folder yet, the demo corpus works fine today")

## Plot it, PCA, then t-SNE (the chart is a choice)

In [ ]:
# TODO (skeleton): write a helper plot_2d(coords, labels, title) that scatter-plots the
# 2-D coordinates and annotates each point with its label.

# TODO: reduce `vecs` to 2-D with PCA and plot it.

# TODO: reduce `vecs` to 2-D with t-SNE and plot it. Compare the two pictures.

## Hunt the surprise cluster

"Near" means "probably similar"; exact distances mean little.

In [ ]:
# TODO: for one item, find its nearest neighbours by cosine similarity (vecs are unit-norm,
# so a dot product IS cosine similarity). Print the item and its neighbours.

## Demo: search by meaning, not by words

Watch this first. The queries below share **no words** with the corpus, and the map still finds
the right neighborhood, because query and corpus were both placed by meaning. A keyword search
returns nothing here; the vector search returns the right cluster.

In [ ]:
import numpy as np

def search(query, k=3):
    qv = embed_text([query])[0]
    sims = (vecs @ qv) / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(qv) + 1e-9)
    print("query:", query)
    for j in sims.argsort()[::-1][:k]:
        print(f"  {sims[j]:.2f}  {corpus[j]}")
    print()

search("outer space")       # no shared words with the rocket sentences
search("a broken heart")    # no shared words with the heartbreak lyrics

### Your turn: drop your own sentence onto the map

Write one sentence, a lyric you love, a line from your own corpus, anything, and see which
cluster claims it. Then try to write one that lands *between* clusters; the starred point on the
re-drawn map shows where yours fell.

In [ ]:
your_sentence = "edit me: write one sentence of your own, then run this cell"

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

yv = embed_text([your_sentence])[0]
sims = (vecs @ yv) / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(yv) + 1e-9)
print("nearest to yours:")
for j in sims.argsort()[::-1][:3]:
    print(f"  {sims[j]:.2f}  {corpus[j]}")

coords = PCA(n_components=2).fit_transform(np.vstack([vecs, yv]))
plt.figure(figsize=(6,5))
plt.scatter(coords[:-1,0], coords[:-1,1])
plt.scatter(coords[-1,0], coords[-1,1], marker="*", s=220)
for (x, y), lab in zip(coords[:-1], corpus):
    plt.annotate(lab[:18], (x, y), fontsize=7)
plt.annotate("YOURS", coords[-1], fontsize=9, weight="bold")
plt.title("Your sentence joins the map"); plt.tight_layout(); plt.show()

## The image path, the same map, for pictures

Image projects embed *pictures* with a CLIP model (`clip-ViT-B-32`) and watch them group by
visual style nobody tagged.

In [ ]:
# TODO: build a handful of images (or load your own), embed them with CLIP
# (sentence-transformers "clip-ViT-B-32"), reduce to 2-D, and scatter-plot them.
# The shape is identical to the text path above; only the embedder changes.

## You made a map of meaning

Your corpus placed itself by meaning, and you saw the same data tell two stories under PCA vs. t-SNE.

**Sketch:** toggle PCA vs. t-SNE and screenshot how the map changes; name one cluster you believe and one you don't.